In [3]:
# Imports
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import random
import sleap_io as sio
import glob

## Pipeline Overview

#### 1. Inspect `.slp` Predictions

- Load each `.slp` file and confirm `video.shape[0] = T`.
- Note that predictions are **sparse** in `labels.labeled_frames` — record the count plus min/max `frame_idx`.
- Compute pairwise frame overlap sets (e.g. `Top ∩ North`, `Top ∩ South`, etc.).

#### 2. Extract Dense Per-Frame Arrays *(per camera)*

Create arrays over the full timeline `0..T-1`:

| Array | Shape | Description |
|---|---|---|
| `coords` | `(T, 4, 2)` | Corner coordinates in fixed order `[corner_1 … corner_4]` |
| `scores` | `(T, 4)` | Confidence scores per corner |
| `frame_has_prediction` | `(T,)` | Boolean mask — frame has at least one prediction |

Use `NaN` for missing data.

#### 3. Quality Filter

Define `frame_valid` of shape `(T,)`:

- Keep frames where **all 4 corners** are present.
- Keep frames where **all scores ≥ threshold** (e.g. `0.85`).
- Set `coords` / `scores` to `NaN` for invalid frames.

#### 4. Pair Frames *(Top ↔ each side camera)*

For each side camera, compute:

```python
matched_frames = where(Top.frame_valid & Side.frame_valid)
```

#### 5. Build + Save Mappings *(for RANSAC homography)*

For each side camera, gather the following arrays:

| Variable | Shape |
|---|---|
| `frames` | `(N,)` |
| `top_pts` | `(N, 4, 2)` |
| `side_pts` | `(N, 4, 2)` |

Save to a compact `.npz` format.

> **Next step:** fit a planar mapping (homography) per side camera using **RANSAC**.

***Nest camera is excluded due to poor confidence in sleap labelling score, and can be added easily manually :)***

### Metadata

In [58]:
# Experiment params, paths and constants
experiment = 'abcEphysPilot01'
session = '2026-04-14T141851Z_calibrateBeforeAprilExperiments'
arena = 'AEON3'
chunk = '2026-04-14T14-00-00'

PRED_DIR = Path(
    "/Users/zosiasus/Documents/Aeon3_SLEAP/aeon3_data/"
    f"{experiment}_sleap_for_card/models/card_inference_results/predictions"
)

CAMERAS = ["CameraTop", "CameraNorth", "CameraSouth", "CameraEast", "CameraWest"]
NODE_ORDER = ["corner_1", "corner_2", "corner_3", "corner_4"] # i think this may be also extracted from the skeleton file :P
conf_threshold = 0.85 # this can be adjusted depending on the size and quality of the SLEAP data

# Where you want outputs to live
OUT_DIR = Path(f"/Users/zosiasus/Documents/Aeon3_SLEAP/frame_mapping_output/{experiment}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Get a view of the data in the .slp files

In [52]:
def load_labels_by_camera(pred_dir: Path, cameras=CAMERAS):
    paths = {cam: next(pred_dir.glob(f"{cam}_*.slp")) for cam in cameras}
    labels = {cam: sio.load_file(p) for cam, p in paths.items()}
    return paths, labels

def inspect_slp_set(labels_by_cam: dict) -> pd.DataFrame:
    """
    Summarize what’s inside each .slp:
    - total video frames (T)
    - sparse prediction count (n_labeled_frames)
    - frame_idx min/max
    - checks: instances per frame, points per instance -> should be 1 instance per frame!
    """
    rows = []
    frame_sets = {}

    for cam, labels in labels_by_cam.items():
        v = labels.videos[0]
        T = int(v.shape[0])  # total frames in the underlying video
        idxs = np.array([lf.frame_idx for lf in labels.labeled_frames], dtype=int)
        frame_sets[cam] = set(idxs.tolist())

        # quick structural sanity on a prefix (fast, enough to validate assumptions)
        bad = 0
        for lf in labels.labeled_frames[:2000]:
            if len(lf.instances) != 1:
                bad += 1
                continue
            if len(lf.instances[0].points) != 4:
                bad += 1

        rows.append(
            {
                "camera": cam,
                "video_T": T,
                "video_shape": tuple(v.shape),
                "n_labeled_frames": len(labels.labeled_frames),
                "first_instance_frame_idx": int(idxs.min()) if idxs.size else None,
                "last_instance_frame_idx": int(idxs.max()) if idxs.size else None,
                "unique_frame_idxs": int(np.unique(idxs).size),
                "bad_frames_first2000": bad,
                "video_filename_in_slp": str(v.filename),
            }
        )

    df = pd.DataFrame(rows).sort_values("camera")
    return df, frame_sets

def inspect_overlaps(frame_sets: dict, ref="CameraTop") -> pd.DataFrame:
    """
    Pairwise overlap (Top ∩ Side) and also global intersection/union sizes.
    """
    cams = list(frame_sets.keys())
    all_intersection = set.intersection(*(frame_sets[c] for c in cams)) if cams else set()

    rows = [{
        "pair": "ALL",
        "intersection_size": len(all_intersection),
    }]

    for cam in cams:
        if cam == ref:
            continue
        inter = frame_sets[ref] & frame_sets[cam]
        rows.append(
            {
                "pair": f"{ref} ∩ {cam}",
                "intersection_size": len(inter),
                "ref_only_missing_in_other": len(frame_sets[ref] - frame_sets[cam]),
                "other_only_missing_in_ref": len(frame_sets[cam] - frame_sets[ref]),
            }
        )
    return pd.DataFrame(rows)

def inspect_prediction_stats(labels_by_cam: dict, node_order=NODE_ORDER, min_score: float = 0.85) -> pd.DataFrame:
    rows = []

    for cam, labels in labels_by_cam.items():
        # counts over labeled_frames
        n_lf = len(labels.labeled_frames)
        inst_counts = []
        point_counts = []

        # per-node score collection
        scores_by_node = {n: [] for n in node_order}
        any_scores = []

        # count how many points we see by name
        name_counts = {n: 0 for n in node_order}

        for lf in labels.labeled_frames:
            inst_counts.append(len(lf.instances))
            if len(lf.instances) < 1:
                continue

            inst = lf.instances[0]
            pts = np.asarray(inst.points)
            point_counts.append(len(pts))

            for pt in pts:
                name = pt["name"]
                score = float(pt["score"])
                any_scores.append(score)
                if name in scores_by_node:
                    scores_by_node[name].append(score)
                    name_counts[name] += 1

        inst_counts = np.array(inst_counts, dtype=int) if inst_counts else np.array([], dtype=int)
        point_counts = np.array(point_counts, dtype=int) if point_counts else np.array([], dtype=int)
        any_scores = np.array(any_scores, dtype=float) if any_scores else np.array([], dtype=float)

        row = {
            "camera": cam,
            "n_labeled_frames": n_lf,
            "instances_per_frame_max": int(inst_counts.max()) if inst_counts.size else None,
            "points_per_instance_min": int(point_counts.min()) if point_counts.size else None,
            "points_per_instance_max": int(point_counts.max()) if point_counts.size else None,
            # "points_per_instance_mean": float(point_counts.mean()) if point_counts.size else None,
            "mean_score_all_points": float(any_scores.mean()) if any_scores.size else None,
            "frac_points_above_thresh": float((any_scores >= min_score).mean()) if any_scores.size else None,
        }

        # add per-corner mean score + counts
        for n in node_order:
            s = np.array(scores_by_node[n], dtype=float)
            row[f"{n}_count"] = int(name_counts[n])
            row[f"{n}_mean_score"] = float(s.mean()) if s.size else None
            row[f"{n}_frac_points_above_thresh"] = float((s >= min_score).mean()) if s.size else None

        rows.append(row)

    return pd.DataFrame(rows).sort_values("camera")

# --- run ---
paths, labels_by_cam = load_labels_by_camera(PRED_DIR, CAMERAS)

summary_df, frame_sets = inspect_slp_set(labels_by_cam)
overlap_df = inspect_overlaps(frame_sets, ref="CameraTop")
stats_df = inspect_prediction_stats(labels_by_cam, min_score=conf_threshold)


print("\nSUMMARY OF DATA SHAPE AND LABELLING FROM SLEAP")
display(summary_df)
print("\nSUMMARY OF INSTANCE OVERLAP BETWEEN CAMERAS")
display(overlap_df)
print("\nSUMMARY OF INSTANCE #, LABEL # & QUALITY OF LABELLING")
print(f"Confidence score threshold set for: {conf_threshold}")
display(stats_df)


SUMMARY OF DATA SHAPE AND LABELLING FROM SLEAP


,camera,video_T,video_shape,n_labeled_frames,first_instance_frame_idx,last_instance_frame_idx,unique_frame_idxs,bad_frames_first2000,video_filename_in_slp
3,CameraEast,29584,"(29584, 1080, 1440, 1)",8148,1844,27319,8148,0,/Volumes/aeon/aeon/data/raw/AEON3/abcEphysPilo...
1,CameraNorth,29584,"(29584, 1080, 1440, 1)",8272,383,26548,8272,0,/Volumes/aeon/aeon/data/raw/AEON3/abcEphysPilo...
2,CameraSouth,29584,"(29584, 1080, 1440, 1)",6797,6093,28027,6797,0,/Volumes/aeon/aeon/data/raw/AEON3/abcEphysPilo...
0,CameraTop,29584,"(29584, 1080, 1440, 1)",18168,386,28047,18168,0,/Volumes/aeon/aeon/data/raw/AEON3/abcEphysPilo...
4,CameraWest,29584,"(29584, 1080, 1440, 1)",8008,2123,27948,8008,0,/Volumes/aeon/aeon/data/raw/AEON3/abcEphysPilo...



SUMMARY OF INSTANCE OVERLAP BETWEEN CAMERAS


,pair,intersection_size,ref_only_missing_in_other,other_only_missing_in_ref
0,ALL,0,NaN,NaN
1,CameraTop ∩ CameraNorth,7955,10213.0,317.0
2,CameraTop ∩ CameraSouth,6357,11811.0,440.0
3,CameraTop ∩ CameraEast,7842,10326.0,306.0
4,CameraTop ∩ CameraWest,7711,10457.0,297.0



SUMMARY OF INSTANCE #, LABEL # & QUALITY OF LABELLING
Confidence score threshold set for: 0.85


,camera,n_labeled_frames,instances_per_frame_max,points_per_instance_min,points_per_instance_max,mean_score_all_points,frac_points_above_thresh,corner_1_count,corner_1_mean_score,corner_1_frac_points_above_thresh,corner_2_count,corner_2_mean_score,corner_2_frac_points_above_thresh,corner_3_count,corner_3_mean_score,corner_3_frac_points_above_thresh,corner_4_count,corner_4_mean_score,corner_4_frac_points_above_thresh
3,CameraEast,8148,1,4,4,0.816053,0.692102,8148,0.926561,0.809524,8148,0.782571,0.669980,8148,0.739683,0.592170,8148,0.815398,0.696735
1,CameraNorth,8272,1,4,4,0.807066,0.688135,8272,0.881524,0.734526,8272,0.809877,0.735131,8272,0.756886,0.630561,8272,0.779977,0.652321
2,CameraSouth,6797,1,4,4,0.775616,0.579153,6797,0.851499,0.648669,6797,0.752706,0.456672,6797,0.730982,0.615566,6797,0.767277,0.595704
0,CameraTop,18168,1,4,4,0.835069,0.633408,18168,0.833748,0.605130,18168,0.824586,0.578435,18168,0.794977,0.583388,18168,0.886965,0.766678
4,CameraWest,8008,1,4,4,0.831188,0.704296,8008,0.907821,0.757118,8008,0.827178,0.699675,8008,0.767619,0.647977,8008,0.822134,0.712413


## Data extraction for pairing of Top and Side camera frames

In [53]:
NODE_TO_I = {n: i for i, n in enumerate(NODE_ORDER)} # just to get dict to change to 0-idx

In [60]:
def extract_card_timeseries(
    slp_path: str | Path,
    min_score: float = 0.85,
    require_all_corners: bool = True,
):
    """
    Convert one SLEAP prediction .slp into dense per-frame arrays.

    Returns
    -------
    out : dict with:
      - coords: (T, 4, 2) float, NaN where missing/filtered -> there will be many missing frames as the card is moved manually ;)
      - scores: (T, 4) float, NaN where missing/filtered -> confidence score for each guessed point
      - frame_has_prediction: (T,) bool   (frame exists in labeled_frames)
      - frame_valid: (T,) bool            (passes your quality rule) -> does it have all corners? Is it a confident score?
      - video_shape: tuple -> frame size (top frame is large, side are smaller)
      - slp_path: Path
    """
    slp_path = Path(slp_path)
    labels = sio.load_file(slp_path)
    v = labels.videos[0]
    T = int(v.shape[0])

    coords = np.full((T, 4, 2), np.nan, dtype=float)
    scores = np.full((T, 4), np.nan, dtype=float)
    frame_has_prediction = np.zeros((T,), dtype=bool)

    # Fill arrays at frame indices where SLEAP stored predictions.
    for lf in labels.labeled_frames:
        t = int(lf.frame_idx)
        if t < 0 or t >= T:
            continue

        frame_has_prediction[t] = True

        # Your data appears to have exactly 1 instance per frame.
        if len(lf.instances) < 1:
            continue
        inst = lf.instances[0]

        # Map the 4 named points into fixed order.
        # Note: inst.points is a structured array with fields: xy, score, visible, complete, name
        for pt in np.asarray(inst.points):
            name = pt["name"]
            if name not in NODE_TO_I:
                continue
            i = NODE_TO_I[name]
            coords[t, i, :] = pt["xy"]
            scores[t, i] = float(pt["score"])

    # Apply confidence masking:
    # - If require_all_corners=True: keep only frames where ALL 4 corners have finite coords AND score>=min_score
    # - Else: you can keep per-corner data where that corner passes min_score
    finite_xy = np.isfinite(coords).all(axis=2)  # (T,4) True if both x and y are finite
    passes_score = np.isfinite(scores) & (scores >= min_score)  # (T,4)

    if require_all_corners:
        frame_valid = (finite_xy & passes_score).all(axis=1)  # (T,)
        # wipe anything not valid
        coords[~frame_valid, :, :] = np.nan
        scores[~frame_valid, :] = np.nan
    else:
        # per-corner masking: corners below threshold are NaN, but others can remain
        keep = finite_xy & passes_score  # (T,4)
        coords[~keep, :] = np.nan
        scores[~keep] = np.nan
        frame_valid = keep.any(axis=1)

    return {
        "slp_path": slp_path,
        "video_shape": tuple(v.shape),
        "coords": coords,
        "scores": scores,
        "frame_has_prediction": frame_has_prediction,
        "frame_valid": frame_valid,
        "node_order": NODE_ORDER,
    }

def build_correspondences(top_ts: dict, side_ts: dict, frames: np.ndarray):
    # Pull the 4 corners for each matched frame
    top_pts = top_ts["coords"][frames]   # (N,4,2)
    side_pts = side_ts["coords"][frames] # (N,4,2)

    # check shape (# of corners)
    ok = np.isfinite(top_pts).all(axis=(1,2)) & np.isfinite(side_pts).all(axis=(1,2))
    frames_ok = frames[ok]
    return frames_ok, top_pts[ok], side_pts[ok]

In [ ]:
top_slp = next(PRED_DIR.glob("CameraTop*.slp"))
top_ts = extract_card_timeseries(top_slp, min_score=conf_threshold, require_all_corners=True)

side_cameras = CAMERAS[1:]

corr = {}  # corr[cam] = {"frames":..., "top_pts":..., "side_pts":...}

for cam in side_cameras:
    slp_path = next(PRED_DIR.glob(f"{cam}*.slp"))
    side_ts = extract_card_timeseries(slp_path, min_score=conf_threshold, require_all_corners=True)

    matched = np.where(top_ts["frame_valid"] & side_ts["frame_valid"])[0]
    frames_ok, top_pts, side_pts = build_correspondences(top_ts, side_ts, matched)

    corr[cam] = {"frames": frames_ok, "top_pts": top_pts, "side_pts": side_pts}

    print(f"\nTop-{cam} usable matched frames: {len(frames_ok)}")
    print("First 10 frame indices:", frames_ok[:10])

# Save everything in one compressed file (keys become cam__frames, cam__top_pts, cam__side_pts)
save_dict = {}
for cam, d in corr.items():
    save_dict[f"{cam}__frames"] = d["frames"]
    save_dict[f"{cam}__top_pts"] = d["top_pts"]
    save_dict[f"{cam}__side_pts"] = d["side_pts"]

out_path = OUT_DIR / f"mapping_top_to_sides_minScore{conf_threshold:.2f}.npz"
np.savez_compressed(out_path, **save_dict)
print("Saved:", out_path)


Top-CameraNorth usable matched frames: 1897
First 10 frame indices: [1298 1696 1697 1698 1699 1700 1701 1883 1884 1885]

Top-CameraSouth usable matched frames: 1039
First 10 frame indices: [7220 7221 7222 7223 7224 7225 7226 7227 7228 7229]

Top-CameraEast usable matched frames: 2281
First 10 frame indices: [1878 1879 1880 1881 1882 1883 1884 1885 1886 1887]

Top-CameraWest usable matched frames: 1767
First 10 frame indices: [2130 2131 2132 2133 2134 2135 2136 2137 2138 2139]
Saved: /Users/zosiasus/Documents/Aeon3_SLEAP/frame_mapping_output/abcEphysPilot01/mapping_top_to_sides_minScore0.85.npz


In [ ]:
## uncomment to load file
# z = np.load("mapping_top_to_sides_minScore0.85.npz")
# north_frames = z["CameraNorth__frames"]
# north_top_pts = z["CameraNorth__top_pts"]     # (N,4,2)
# north_side_pts = z["CameraNorth__side_pts"]   # (N,4,2)